# 📊 Clustering Metrics - Product Recommendation Dataset

<a href="https://colab.research.google.com/github/rubenfonnegra/machine_learning/blob/master/Sem_08/clustering_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/machine_learning/blob/master/Sem_08/clustering_metrics.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 

This notebook compares **K-Means**, **Mean Shift**, and **DBSCAN** using two families of metrics.

*Internal metrics — Are the clusters compact and well separated?*
- Within-Cluster Sum of Squares (WCSS / inertia)
- Silhouette Score

*External metrics — How closely does clustering agree with a known partition?*
- Adjusted Rand Index (ARI)
- Normalized Mutual Information (NMI)


### Learning objectives
- Create train/test partitions.
- Preprocess numerical and categorical features.
- Train three clustering algorithms.
- Compute internal and external clustering metrics.
- Compare the models in tables and plots.


### Documentation

- [```silhouette_score```](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html#sklearn.metrics.silhouette_score) 
- [```adjusted_rand_score```](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html#sklearn.metrics.adjusted_rand_score) 
- [```normalized_mutual_info_score```](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.normalized_mutual_info_score.html#sklearn.metrics.normalized_mutual_info_score) 

---

> **📘 Machine Learning**  
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution

## Imports

In [ ]:
#@markdown #### **🛠️⚙️📦 Install complementary dependencies**. 

from tqdm.auto import tqdm
import subprocess, time, sys

LIB = "MLTools-1.2-py3-none-any.whl"
URL = "https://drive.google.com/uc?id=18Y834Tvtj_-Px9yNmbAB4yV4L0gcIZ20"

commands = [
    ("📦 Downloading resources", ["gdown", URL, "-O", LIB], 35),
    ("🔧 Installing dependencies", [sys.executable, "-m", "pip", "install", "-q", LIB], 55),
    ("🧹 Finishing", ["rm", "-f", LIB], 10)
]

print("⚙️ Iniciando configuración del entorno...\n")

try:
    with tqdm(total=100, desc="Preparando", bar_format="{desc}: {bar} {n:.0f}%") as bar:
        for label, command, weight in commands:
            bar.set_description(label)
            subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            for _ in range(weight):
                time.sleep(0.01)
                bar.update(1)

    print("\n✅ Configuración completada correctamente.")

except subprocess.CalledProcessError as e:
    print("\n❌ Error durante la configuración")
    print(f"Exit code: {e.returncode}")

    if e.stdout:
        print("\n📤 STDOUT:")
        print(e.stdout)

    if e.stderr:
        print("\n🔍 STDERR:")
        print(e.stderr)

    print("\n❌ No fue posible configurar el entorno. Ejecute nuevamente la celda.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, MeanShift, DBSCAN, estimate_bandwidth
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from MLTools import compute_inertia, silhouette_without_noise, dbscan_predict

## Load the dataset

In [ ]:
CSV_PATH = _ 
data = pd.read_csv(CSV_PATH)

expected_columns = [
    "user_id", "age", "gender", "location", "income_level",
    "product_id", "product_category", "product_price",
    "product_popularity", "interaction_type", "rating",
    "time_spent_on_product", "user_avg_spending",
    "user_activity_level", "purchase_frequency",
    "interaction_year", "interaction_month", "interaction_day",
    "recommendation_label"
]

missing = [c for c in expected_columns if c not in data.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

data = data[expected_columns].copy()
print("Shape:", data.shape)
data.head()

## Inspect the dataset

In [ ]:
data.describe(include='all').T

In [ ]:
data.isna().sum()

## Define features and reference labels

`recommendation_label` is the **known/reference partition** used only for external evaluation. It is never supplied to the clustering algorithms.

`user_id` and `product_id` are also excluded because numerical distances between identifiers have no useful clustering meaning.

In [ ]:
target = "recommendation_label"
identifiers = ["user_id", "product_id"]

features = [
    c for c in data.columns
    if c not in identifiers + [target]
]

X = data[features].copy()
y = data[target].copy()

print("Features:", len(features))
print("Reference groups:", y.nunique())
y.value_counts()

## Holdout partition

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

## Preprocessing

Numerical variables are standardized. Nominal categorical variables are one-hot encoded.

The transformer is fitted only on training data and then applied to test data.

In [ ]:
categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

In [ ]:
preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train:", X_train_processed.shape)
print("Test:", X_test_processed.shape)

## Fixed model parameters

This notebook focuses on **evaluation metrics**, so we use fixed parameters. These can later be replaced by values selected using elbow, silhouette, bandwidth, or k-distance analyses.

For K-Means, the number of reference groups is used only as a convenient experimental value; the reference labels themselves are never passed to the model.

In [ ]:
N_CLUSTERS = y_train.nunique()

MEANSHIFT_BANDWIDTH = estimate_bandwidth(
    X_train_processed,
    quantile=0.20,
    n_samples=min(1000, len(X_train_processed))
)

DBSCAN_EPS = 2.5
DBSCAN_MIN_SAMPLES = max(5, 1 * X_train_processed.shape[1])

print("K-Means k:", N_CLUSTERS)
print("Mean Shift bandwidth:", round(MEANSHIFT_BANDWIDTH, 3))
print("DBSCAN eps:", DBSCAN_EPS)
print("DBSCAN min_samples:", DBSCAN_MIN_SAMPLES)

## Train K-Means

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42)

kmeans.fit(X_train_processed)

kmeans_train_labels = kmeans.labels_
kmeans_test_labels = kmeans.predict(X_test_processed)

print("Clusters:", len(np.unique(kmeans_train_labels)))

## Train Mean Shift

In [ ]:
meanshift = MeanShift(bandwidth=MEANSHIFT_BANDWIDTH)

meanshift.fit(X_train_processed)

meanshift_train_labels = meanshift.labels_
meanshift_test_labels = meanshift.predict(X_test_processed)

print("Clusters:", len(np.unique(meanshift_train_labels)))

## Train DBSCAN

Scikit-learn's DBSCAN has no standard `predict()` method. We fit it on training data and use a simple out-of-sample rule for the test set:

1. Find the nearest training core sample.
2. If it is within `eps`, inherit its cluster.
3. Otherwise label the test observation as noise (`-1`).

This is a practical approximation for holdout evaluation, **not part of the original DBSCAN algorithm**.

In [ ]:
dbscan = DBSCAN(
    eps=DBSCAN_EPS,
    min_samples=DBSCAN_MIN_SAMPLES,
    metric="euclidean"
)

dbscan_train_labels = dbscan.fit_predict(X_train_processed)

non_noise = dbscan_train_labels[dbscan_train_labels != -1]

print("Clusters:", len(np.unique(non_noise)))
print("Noise fraction:", round(np.mean(dbscan_train_labels == -1), 3))

dbscan_test_labels = dbscan_predict(dbscan, X_test_processed)

print("Test noise fraction:", round(np.mean(dbscan_test_labels == -1), 3))

## Part II — Internal Metrics

### WCSS / Inertia

Within-Cluster Sum of Squares measures compactness:

$$
WCSS =
\sum_{k=1}^{K}
\sum_{\mathbf{x}_i \in C_k}
\|\mathbf{x}_i-\boldsymbol{\mu}_k\|^2
$$

**Lower is more compact.**

For K-Means, training WCSS is its familiar `inertia_`.

WCSS naturally decreases as the number of clusters increases, so it should **not be used alone to rank models that produce different numbers of clusters**.

DBSCAN noise points are excluded because they do not belong to a cluster.

### Silhouette Score

For each observation:

$$
s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}
$$

where \(a(i)\) represents cohesion and \(b(i)\) separation.

- Near **1** → compact and separated.
- Near **0** → overlapping clusters.
- Negative → potentially inappropriate assignments.

DBSCAN noise is excluded from this internal metric.

## Part III — External Metrics

### Adjusted Rand Index (ARI)

ARI compares whether pairs of observations are grouped similarly in two partitions.

$$
ARI \in [-1,1]
$$

- **1** → perfect agreement.
- Around **0** → agreement comparable to chance.
- Negative → worse than expected by chance.

ARI is **permutation invariant**: cluster IDs do not need to match the numeric values of the reference labels.

### Normalized Mutual Information (NMI)

NMI measures the information shared by the discovered clustering and known partition.

$$
NMI \in [0,1]
$$

- **1** → perfect correspondence.
- **0** → no mutual information.

NMI is also permutation invariant.

## Compute the metrics on the test partition

For DBSCAN:

- noise is excluded from WCSS and silhouette,
- noise is retained as label `-1` for ARI and NMI.

This prevents a model from receiving artificially favorable external evaluation simply by discarding difficult observations as noise.

In [ ]:
model_labels = {
    "K-Means": kmeans_test_labels,
    "Mean Shift": meanshift_test_labels,
    "DBSCAN": dbscan_test_labels
}

results = []

for model_name, labels in model_labels.items():
    regular = labels[labels != -1]

    results.append({
        "model": model_name,
        "n_clusters": len(np.unique(regular)),
        "noise_fraction": np.mean(labels == -1),
        "inertia": compute_inertia( X_test_processed, labels ),
        "silhouette": silhouette_without_noise( X_test_processed, labels ),
        "ari": adjusted_rand_score( y_test, labels ),
        "nmi": normalized_mutual_info_score( y_test, labels )
    })

metrics_df = pd.DataFrame(results).set_index("model")

metrics_df.round(4)

## Internal metrics table

In [ ]:
metrics_df[
    [
        "n_clusters",
        "noise_fraction",
        "inertia",
        "silhouette"
    ]
].round(4)

## External metrics table

In [ ]:
metrics_df[
    [
        "ari",
        "nmi"
    ]
].round(4)

## Silhouette comparison

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(metrics_df.index, metrics_df["silhouette"])

plt.ylabel("Silhouette Score")
plt.xlabel("Model")
plt.title("Internal Evaluation — Silhouette")
plt.show()

## ARI and NMI comparison

In [ ]:
metrics_df[ ["ari", "nmi"] ].plot(kind="bar", figsize=(9, 5))

plt.ylabel("Score")
plt.xlabel("Model")
plt.title("External Evaluation — ARI and NMI")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(title="Metric")
plt.show()

## Complete comparison

In [ ]:
comparison = metrics_df.rename(
    columns={
        "n_clusters": "Clusters",
        "noise_fraction": "Noise Fraction",
        "wcss": "WCSS",
        "silhouette": "Silhouette",
        "ari": "ARI",
        "nmi": "NMI"
    }
)

comparison.round(4)

## Interpreting the results

There is no single clustering metric that answers every question.

### Internal evaluation
Use it when the true grouping is unknown.

- **WCSS:** How compact are the clusters?
- **Silhouette:** Are observations cohesive within clusters and separated from neighboring clusters?

### External evaluation
Use it when a trusted reference partition exists.

- **ARI:** Do the two partitions group pairs similarly?
- **NMI:** How much information do the partitions share?

A clustering can have strong internal structure but weak agreement with the known labels. It may simply have discovered a **different structure** in the data.

Likewise, high external agreement does not guarantee that every cluster is geometrically compact.

## Practice exercises

1. Change K-Means \(k\) and observe WCSS and silhouette.
2. Change Mean Shift bandwidth and recompute all four metrics.
3. Change DBSCAN `eps` while holding `min_samples` fixed.
4. Explain why WCSS should not be compared without considering cluster count.
5. Compare DBSCAN silhouette with different noise fractions.
6. Find a configuration with strong silhouette but weaker ARI/NMI and interpret it.

In [ ]:
# Your code here

---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Machine Learning courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *Clustering metrics: Machine Learning Notebook*. Institución Universitaria Pascual Bravo.